# 第8课：从零实现自动微分

**学习目标：**
- 理解自动微分的核心思想：计算图 + 链式法则
- 实现 `Value` 类，支持自动梯度计算
- 用面向对象方式构建神经网络（Neuron → Layer → MLP）
- 理解 PyTorch `autograd` 的底层原理

---

前面的课程中，我们手动推导了反向传播的每一步。现代框架（PyTorch、TensorFlow）用**自动微分**来自动计算梯度。本课将从零实现一个微型自动微分引擎，帮助你理解其核心原理。

> 本课思路来自 Andrej Karpathy 的 [micrograd](https://github.com/karpathy/micrograd)。

## 8.1 Value 类：带梯度的数值

`Value` 类不仅存储数值，还自动构建**计算图**，记录每个运算的依赖关系。

核心属性：
- `data`：存储实际数值
- `grad`：存储梯度（偏导数）
- `_prev`：父节点集合，形成计算图
- `_backward`：该节点的梯度计算函数

In [ ]:
import numpy as np
import math

class Value:
    """带自动微分的标量值"""
    
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = float(data)
        self.grad = 0.0              # 梯度初始为0
        self._backward = lambda: None  # 默认无操作
        self._prev = set(_children)    # 父节点
        self._op = _op                 # 运算符
        self.label = label
    
    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"
    
    def __add__(self, other):
        """加法：z = a + b"""
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        
        def _backward():
            self.grad += out.grad   # dz/da = 1
            other.grad += out.grad  # dz/db = 1
        out._backward = _backward
        return out
    
    def __mul__(self, other):
        """乘法：z = a * b"""
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        
        def _backward():
            self.grad += other.data * out.grad   # dz/da = b
            other.grad += self.data * out.grad   # dz/db = a
        out._backward = _backward
        return out
    
    def __pow__(self, other):
        """幂运算：z = a^n"""
        assert isinstance(other, (int, float))
        out = Value(self.data ** other, (self,), f'**{other}')
        
        def _backward():
            self.grad += other * (self.data ** (other - 1)) * out.grad
        out._backward = _backward
        return out
    
    def relu(self):
        """ReLU 激活"""
        out = Value(max(0, self.data), (self,), 'ReLU')
        
        def _backward():
            self.grad += (1.0 if self.data > 0 else 0.0) * out.grad
        out._backward = _backward
        return out
    
    def backward(self):
        """反向传播：计算整个图的梯度"""
        # 拓扑排序：确保依赖顺序正确
        topo = []
        visited = set()
        
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        
        build_topo(self)
        
        # 从后往前计算梯度
        self.grad = 1.0  # 输出的梯度为1
        for node in reversed(topo):
            node._backward()
    
    # 支持反向运算
    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)
    def __truediv__(self, other): return self * other**-1
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other
    def __rsub__(self, other): return Value(other) - self

## 8.2 自动微分演示

看看自动微分如何工作：前向传播自动构建计算图，`backward()` 一键计算所有梯度。

In [ ]:
# 构建一个简单的计算图
a = Value(2.0, label='a')
b = Value(-3.0, label='b')
c = Value(10.0, label='c')
e = a * b; e.label = 'e'        # e = a * b = -6
d = e + c; d.label = 'd'        # d = e + c = 4
f = Value(-2.0, label='f')
L = d * f; L.label = 'L'        # L = d * f = -8

print(f"L = {L.data}")

# 一键反向传播
L.backward()

print(f"\ndL/da = {a.grad}")  # dL/da = b*f = -3*-2 = 6
print(f"dL/db = {b.grad}")  # dL/db = a*f = 2*-2 = -4
print(f"dL/dc = {c.grad}")  # dL/dc = f = -2
print(f"dL/df = {f.grad}")  # dL/df = d = 4

## 8.3 用 Value 构建神经网络

从底层到高层逐步抽象：Neuron → Layer → MLP

In [ ]:
import random

class Neuron:
    """单个神经元：sum(wi*xi) + b → ReLU"""
    def __init__(self, n_inputs):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(n_inputs)]
        self.b = Value(0)
    
    def __call__(self, x):
        # 加权和 + 偏置
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.relu()
    
    def parameters(self):
        return self.w + [self.b]

class Layer:
    """一层多个神经元"""
    def __init__(self, n_inputs, n_neurons):
        self.neurons = [Neuron(n_inputs) for _ in range(n_neurons)]
    
    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs
    
    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]

class MLP:
    """多层感知机"""
    def __init__(self, n_inputs, layer_sizes):
        sizes = [n_inputs] + layer_sizes
        self.layers = [Layer(sizes[i], sizes[i+1]) for i in range(len(layer_sizes))]
    
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

## 8.4 训练演示

用 MLP 在简单数据集上训练，展示自动微分的威力。

In [ ]:
# 简单二分类数据
np.random.seed(42)
n_samples = 50
X_data = np.random.randn(n_samples, 2).tolist()
y_data = [1 if x[0]**2 + x[1]**2 > 1.5 else -1 for x in X_data]

# 创建 MLP：2维输入 → 8 → 8 → 1维输出
mlp = MLP(2, [8, 8, 1])
print(f"参数总数: {len(mlp.parameters())}")

In [ ]:
# 训练循环
learning_rate = 0.01

for epoch in range(100):
    # 前向传播
    scores = [mlp([Value(x[0]), Value(x[1])]) for x in X_data]
    # 确保输出是标量
    scores = [s[0] if isinstance(s, list) else s for s in scores]
    
    # SVM 损失
    losses = [(1 + -yi * score).relu() for yi, score in zip(y_data, scores)]
    data_loss = sum(losses) * (1.0 / len(losses))
    
    # L2 正则化
    reg_loss = sum((p * p for p in mlp.parameters())) * 0.001
    total_loss = data_loss + reg_loss
    
    # 反向传播
    for p in mlp.parameters():
        p.grad = 0.0
    total_loss.backward()
    
    # 更新
    for p in mlp.parameters():
        p.data -= learning_rate * p.grad
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {total_loss.data:.4f}")

---

## 小结

- **自动微分**：通过计算图自动应用链式法则计算梯度
- **`Value` 类**：存储数值 + 梯度，记录计算依赖
- **拓扑排序**：确保梯度计算的正确顺序
- **`backward()`**：一键计算整个计算图的梯度
- **这正是 PyTorch `autograd` 的核心思想**

至此，NumPy 版教程全部完成。接下来进入 **PyTorch 教程**，你会发现很多概念已经不再陌生。